# ARQWELIA Lot 2 — SDXL Inpainting on a free GPU (benchmark only)

**WARNING**
- GPU availability is NOT guaranteed on free environments.
- A free environment is NOT appropriate for Production.
- Delete temporary files after the session.
- Do NOT use a real user photo during Phase 0A (synthetic benchmark images only).
- Do NOT expose ComfyUI on the Internet; no public tunnel is included.
- Do NOT store any DeepSeek API key in this notebook.
- The image + mask stay local; nothing is sent to a remote provider.
- This notebook aligns with the POC contract: proportional resize + padding,
  the SAME transform applied to the mask (nearest), grayscale mask,
  one generation, then recompose the masked area with the source.
  It must NOT be executed during the CI validation of PR #80 (no model
  download, no installation in CI).

## 1. Verify CUDA GPU presence

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True, timeout=20)
    print(out.stdout or "NO GPU / nvidia-smi not found")
except Exception as exc:
    print("GPU check failed:", exc)


## 2. Install locked dependencies

Versions are pinned for reproducibility. Requires a CUDA-capable runtime (PyTorch).

In [ ]:
import sys
!{sys.executable} -m pip install --quiet torch==2.4.1 diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 pillow==10.4.0
print("dependencies installed")

## 3. Load SDXL Inpainting from its official repository

In [ ]:
import torch
from diffusers import StableDiffusionXLInpaintPipeline

model_id = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.to("cuda" if torch.cuda.is_available() else "cpu")
print("pipeline loaded")

## 4. Load source image + mask and prepare the 1024x1024 working canvas

Strategy (matches the POC contract):
- proportional resize (fit inside 1024x1024, never stretched);
- centered padding to 1024x1024;
- the SAME scale + offsets applied to the grayscale mask (nearest-neighbour);
- the mapping (scale, offsetX, offsetY, original/working dims) is kept.

In [ ]:
from PIL import Image

SOURCE_PATH = "dataset/photos/synthetic01.png"
MASK_PATH = "dataset/masks/synthetic01-pool-mask.png"
WORKING = 1024

img = Image.open(SOURCE_PATH).convert("RGB")
mask = Image.open(MASK_PATH).convert("L")
orig_w, orig_h = img.size
assert mask.size == img.size, "mask and image must be the same size"

scale = min(WORKING / orig_w, WORKING / orig_h)
resized_w = max(1, round(orig_w * scale))
resized_h = max(1, round(orig_h * scale))
offset_x = max(0, (WORKING - resized_w) // 2)
offset_y = max(0, (WORKING - resized_h) // 2)

working_image = Image.new("RGB", (WORKING, WORKING), (0, 0, 0))
working_image.paste(img.resize((resized_w, resized_h)), (offset_x, offset_y))
working_mask = Image.new("L", (WORKING, WORKING), 0)
working_mask.paste(mask.resize((resized_w, resized_h), Image.NEAREST), (offset_x, offset_y))

mapping = {
    "scale": scale,
    "offsetX": offset_x,
    "offsetY": offset_y,
    "originalWidth": orig_w,
    "originalHeight": orig_h,
    "workingWidth": WORKING,
    "workingHeight": WORKING,
}
print("working image size:", working_image.size, "| mask size:", working_mask.size)
print("mapping:", mapping)

## 5. Use a VisualBrief JSON

In [ ]:
import json

visual_brief = {
    "version": "arqwelia-visual-brief-v1",
    "concept": "A",
    "sceneType": "residential_garden_pool_inpainting",
    "pool": {
        "shape": "rectangular",
        "estimatedDimensions": "8x4m",
        "placement": "central_open_lawn",
        "orientation": "parallel_to_house",
    },
    "preserve": ["house_architecture", "camera_perspective", "boundary_fences", "mature_trees", "unmasked_pixels"],
    "add": ["realistic_in_ground_pool", "natural_stone_coping", "mediterranean_landscaping"],
    "negative": ["people", "text", "logo", "house_distortion", "extra_buildings", "duplicate_pool", "floating_objects", "unrealistic_reflections"],
    "inpaintingPrompt": "Photorealistic in-ground swimming pool added to a residential rear garden. Pool shape: rectangular. Pool dimensions: 8x4m. Garden style: mediterranean. Coping: natural_stone. Terrace: natural_stone_patio. Budget range: medium. Preserve the house, fences, existing trees and the exact camera perspective. Only modify the masked area; keep every unmasked pixel unchanged. Natural lighting, no people, no text, no logos, realistic water reflections.",
    "negativePrompt": "people, faces, text, watermark, logo, distorted architecture, extra buildings, second pool, floating objects, unrealistic reflections, warped geometry, construction equipment, cartoon style, oversaturated colors",
    "recommended": {"steps": 25, "cfg": 7, "strength": 0.82, "seed": 42},
}
print(json.dumps(visual_brief, indent=2)[:600])

## 6. Run ONE manual generation with the working canvas

In [ ]:
import torch

generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(visual_brief["recommended"]["seed"])
result = pipe(
    prompt=visual_brief["inpaintingPrompt"],
    negative_prompt=visual_brief["negativePrompt"],
    image=working_image,
    mask_image=working_mask,
    width=WORKING,
    height=WORKING,
    num_inference_steps=visual_brief["recommended"]["steps"],
    guidance_scale=visual_brief["recommended"]["cfg"],
    strength=visual_brief["recommended"]["strength"],
    generator=generator,
)
print("generation done")

## 7. Recompute the result with the source outside the mask

Masked pixels = generated; unmasked pixels = the exact working source.

In [ ]:
from PIL import ImageChops

generated = result.images[0].convert("RGB")
composite = Image.composite(generated, working_image, working_mask)
print("composite size:", composite.size)

## 8. Save the image + mapping

In [ ]:
import os
os.makedirs("benchmark-out/deepseek-comfyui-poc", exist_ok=True)
composite.save("benchmark-out/deepseek-comfyui-poc/notebook-sdxl-result.png")
with open("benchmark-out/deepseek-comfyui-poc/notebook-mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)
print("saved")

## 9. Print technical metadata (measured output dims)

In [ ]:
print("seed:", visual_brief["recommended"]["seed"])
print("steps:", visual_brief["recommended"]["steps"])
print("cfg:", visual_brief["recommended"]["cfg"])
print("strength:", visual_brief["recommended"]["strength"])
print("working canvas:", working_image.size)
print("composite output size:", composite.size)
print("model:", model_id)

## 10. Stop

Generation complete. Delete temporary files and free the GPU after the session.